### Make dict with item url being the (normalized) URL, and the value being the bibtex citekey

Used for matching URL cites in perplexity dialogs to the bibext citekey filenames of obsidian notes.

In [ ]:
# TODO: 
# - SPACE BEFORE LINKS IN MD DOC, 
# - SOME UNCLOSED '']'' NEAR eol
# - ORIG FOOTNOTES ALSO MISSING
# - RETAIN ORIGINAL CONTENTS, SO CAN CONSIDER ADDING NEW LINKS FROM IT LATER

In [64]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
obsidian_citekeys_file = rfw.refwrangle_test_dir / "dat" / 'obsnotecitekeys.csv'

output_file = tmp_dir / "tmp_new_cites_perplexity_example.md"

In [2]:
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())

In [24]:
# get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes
citekeys = {}
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
for (key, value_list) in citekeysForURL.items():
    url_to_citekey[key] = value_list[0]

In [65]:
lit_note_file_stems = {fNm.stem for fNm in rfw.lit_notes_obsidian_dir.glob('*.md')}
citekeys = {}
zot_db_items = []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)
    zot_db_items.append(dict(citekey=citekeyThis, zotkey=parent['key'], hasLitNote=citekeyThis in lit_note_file_stems))

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
citekey_to_url = {}
for (url, citekey_list) in citekeysForURL.items():
    url_to_citekey[url] = citekey_list[0]
    citekey_to_url[citekey_list[0]] = url


#pd.Series(citekey_to_url)
zot_db_items = pd.DataFrame(zot_db_items).set_index('citekey')
zot_db_items['url'] = pd.Series(citekey_to_url)
zot_db_items.reset_index()

hasNoURL = zot_db_items.url.isna()
if (nURLmiss := sum(hasNoURL)) > 0:
    print(f"Dropping {nURLmiss=} of {len(zot_db_items)} zotero entries which have no URL")
    zot_db_items_no_url = zot_db_items[hasNoURL]
    zot_db_items = zot_db_items[~hasNoURL]
    display(zot_db_items_no_url)


Dropping nURLmiss=165 of 1680 zotero entries which have no URL


,zotkey,hasLitNote,url
citekey,,,
Seals99irradFrcctDiag,WYP9J7EU,False,NaN
LaPaglia13TestIncrsSuggestibility,LDTF7M3L,False,NaN
Gaur20attribModellingRvw,HLHKVCLX,False,NaN
Holloway23emotionCuePolitJudge,4AUIK74F,False,NaN
Tomeo21predElectAgeSocMedia,PX6DNXY2,False,NaN
...,...,...,...
Mayhorn16disaggLdRealWrldPerf,RCMMDNV6,False,NaN
Heinemann06frcstSolRadCMV,NYJFP3I4,False,NaN
Lorenz07frcstEnsGridPV,2C5N2DZC,False,NaN


In [68]:
def zotero_item_link(zotero_item_key, link_text):
    return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

def normalize_url(url):
    """Convert a URL to a standard form, so the it can be string-compared to the same URL
    written by a different program, but which is also normalized by this function."""
    parsed = urlparse(url.lower())
    return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

def replace_perplexity_citations_from_perplexity(markdown_file, zot_db_items, output_file):
    """Replace numeric citations in a perplexity dialog with any obsidian literature note links that are
    given either as a dict or a file."""

    # Read the CSV file and create a dictionary of normalized URL to citekey mappings
    if not isinstance(zot_db_items, pd.DataFrame):
        raise Exception('Expected a dataframe.  Reading url_to_citekey from file does not yet handle new dataframe column')
        df = pd.read_csv(zot_db_items) # assume it has url and citekey columns
        zot_db_items = {normalize_url(url): citekey for url, citekey in zip(df.url, df.citekey)}

    zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)

    # Read the Markdown file
    with open(markdown_file, 'r') as mdfile:
        content = mdfile.read()

    # Split the content into body and citations
    parts = content.split("\nCitations:\n")
    if len(parts) != 2:
        raise Exception("Couldn't find Citations section")
    
    body, citations = parts

    # Extract citations and their corresponding normalized URLs
    citation_urls = re.findall(r'\[(\d+)\]\s+(https?://\S+)', citations)
    doc_url_to_number = {normalize_url(url): num for num, url in citation_urls}

    # Replace citations in the body text with a wikilinks to an obsidian note, or to or zotero item
    def replace_citation(match):
        num = match.group(1)
        doc_url = next((url for url, cite_num in doc_url_to_number.items() if cite_num == num), None)

        if doc_url:
            #itemInfo = zot_db_items[zot_db_items.url == doc_url]
            itemInfo = zot_db_items[zot_db_items.url == doc_url].iloc[0] if not zot_db_items[zot_db_items.url == doc_url].empty else None
            if itemInfo:
                if itemInfo.hasLitNote:
                    return f' [[{itemInfo.citekey}]]'
                link_text = f'{itemInfo.zotkey}={itemInfo.citekey}'
                return f' {zotero_item_link(itemInfo.zotkey, link_text)}'
        return f' [{num}]'

    body = re.sub(r'\[(\d+)\]', replace_citation, body)

    # Replace citations in the Citations section
    def replace_citation_in_references(match):
        num = match.group(1)
        url = match.group(2)
        normalized_url = normalize_url(url)
        if normalized_url in zot_db_items:
            return f'[[{zot_db_items[normalized_url]}]] {url}'
        return f'[{num}] {url}'

    citations = re.sub(r'\[(\d+)\]\s+(https?://\S+)', replace_citation_in_references, citations)

    # Combine modified body and citations
    modified_content = body + "\nCitations:\n" + citations

    # Write the modified content to the output file
    with open(output_file, 'w') as outfile:
        outfile.write(modified_content)


In [69]:
replace_perplexity_citations_from_perplexity(perplexity_dialog_file, zot_db_items, output_file)
#rfw.ORIG_replace_perplexity_citations_from_perplexity(perplexity_dialog_file, url_to_citekey, output_file)
print('Done.')

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().